##IMPORTING THE FILE AND THE LIBRARIES

In [58]:
import numpy as np
import pandas as pd
from plotly.express import scatter, pie

In [59]:
df = pd.read_csv('https://raw.githubusercontent.com/amankharwal/Website-data/master/tips.csv')
df.head()


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [60]:
print(df.shape)
print(df.isnull().sum())

(244, 7)
total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64


##DATA VISUALIZATION

###Total Bill vs Tip

Do bigger bills lead to bigger tips? Yes, tips rise with the bill (correlation ~0.68), but the dots spread out more on the large bills, so the bill alone doesn't fix the tip.

In [61]:
df['total_bill'].corr(df['tip'])

np.float64(0.6757341092113641)

In [62]:
fig = scatter(df, x='total_bill', y='tip', title='Total bill vs Tip')
fig.update_traces(marker=dict(color='rgba(232,111,172,1)'))
fig.update_layout(plot_bgcolor = 'rgba(251, 229, 240, 0.8)')
fig.show()

###Size vs Tip

Do bigger parties tip more? Yes, the average tip goes up with party size, from about 2.58\$ for 2 people to about 4.14\$ for 4. Most parties have 2 people, though, very few have 1, 5, or 6, so the model has little data for those sizes.

In [63]:
df.groupby('size')["tip"].mean()

,tip
size,
1,1.437500
2,2.582308
3,3.393158
4,4.135405
5,4.028000
6,5.225000


In [64]:
df['size'].value_counts()

,count
size,
2,156
3,38
4,37
5,5
1,4
6,4


In [65]:
fig = scatter(df, x='size', y='tip', title='Party Size vs Tip')
fig.update_traces(marker=dict(color='rgba(35, 79, 61, 1)'))
fig.update_layout(plot_bgcolor = 'rgba(183, 201, 176, 0.8)')
fig.show()

###Bill vs Tip, colored by time

Do lunch and dinner behave differently? The two groups follow roughly the same upward pattern, but dinner bills are bigger (average about 20.8\$ vs 17.17\$ at lunch), so dinner tips are slightly higher.

In [66]:
df.groupby('time')["total_bill"].mean()

,total_bill
time,
Dinner,20.797159
Lunch,17.168676


In [67]:
fig = scatter(df, x='total_bill', y='tip', color='time',  color_discrete_map={'Lunch': 'rgba(193, 124, 74, 1)', 'Dinner': 'rgba(139, 114, 181, 1)'}, title='Total bill vs tip, lunch vs dinner')
fig.update_layout(plot_bgcolor = 'rgba(255, 248, 240, 0.8)')
fig.show()

###Days

Which days does the data come from? Mostly Saturday (87 rows), Sunday(76) and Thursday(62) and only 19 rows are from Friday, so the model has the least to learn from Fridays.

In [68]:
df["day"].value_counts()

,count
day,
Sat,87
Sun,76
Thur,62
Fri,19


In [69]:
fig = pie(df, names='day', title='Share of bills by day of the week', color_discrete_sequence=['#F8F1E7', '#E68AB3', '#A8BFA0', '#8E79B8', '#C68A5B'])
fig.show()

###Smokers vs Non-Smokers

Is the data balanced between smokers and non-smokers? Not quite, about 62% of the bills are from non-smokers.

In [70]:
fig = pie(df, names='smoker', title='Smokers vs non-smokers', color_discrete_sequence=['#7C9A92', '#D4A5A5'] )
fig.show()

##PREPARE THE DATA

In [71]:
y = df['tip']
X = df.drop(columns='tip')
X = pd.get_dummies(X, drop_first=True)
print(X.shape, y.shape)
X.head()

(244, 8) (244,)


,total_bill,size,sex_Male,smoker_Yes,day_Sat,day_Sun,day_Thur,time_Lunch
0,16.99,2,False,False,False,True,False,False
1,10.34,3,True,False,False,True,False,False
2,21.01,3,True,False,False,True,False,False
3,23.68,2,True,False,False,True,False,False
4,24.59,4,False,False,False,True,False,False


##IMPORT THE MODEL

In [72]:
from sklearn.linear_model import LinearRegression

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(195, 8) (49, 8)


##TRAIN THE MODEL

In [74]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

##TEST THE MODEL

In [75]:
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f'Accuracy (R²) on training data: {train_score:.3f}')
print(f'Accuracy (R²) on test data: {test_score:.3f}')

Accuracy (R²) on training data: 0.458
Accuracy (R²) on test data: 0.437


In [76]:
with open('results.txt', 'w') as f:
    f.write(f'Training R2: {train_score:.3f}\n')
    f.write(f'Test R2: {test_score:.3f}\n')
    f.write(f'Predicted tip for the example: {prediction:.2f}\n')

In [77]:
example = pd.DataFrame([{'total_bill': 50.0, 'sex': 'Male', 'smoker': 'No','day': 'Sat', 'time': 'Dinner', 'size': 3}])
example = pd.get_dummies(example).reindex(columns=X.columns, fill_value=0)

prediction = model.predict(example)[0]
print(f'Predicted tip: ${prediction:.2f}')

Predicted tip: $6.01


###ACTUAL VS PREDICTED TIPS

Each dot is one test row: the x-axis is the real tip and the y-axis is the tip the model predicted. The closer a dot is to the dashed line, the better the prediction.

In [78]:
y_pred = model.predict(X_test)
results = pd.DataFrame({'Actual tip': y_test, 'Predicted tip': y_pred})

fig = scatter(results, x='Actual tip', y='Predicted tip',
              title='Actual vs predicted tips (test data)', width=650, height=500)
fig.update_traces(marker=dict(color='rgba(35, 79, 61, 1)'))

lim = [results.min().min(), results.max().max()]
fig.add_shape(type='line', x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1],
              line=dict(dash='dash', color='black'))
fig.show()
errors = (results['Actual tip'] - results['Predicted tip']).abs()
print(f'Average error: ${errors.mean():.2f}')
print(f'Predictions within $1 of the real tip: {(errors < 1).mean():.0%}')


Average error: $0.67
Predictions within $1 of the real tip: 80%
